# Implementation of arXiv:2408.04649, "Chain of Stance: Stance Detection with Large Language Models"

This notebook uses LangGraph and Ollama. It is very slow and impractical to use for labelling stance of thousands of comments. 

I concluded it was better to use vLLM but LangChain's integration of vLLM is lacking and you can't send batches only a single message at a time!

In [1]:
from typing import Annotated, TypedDict, Dict
from langchain_core.messages import BaseMessage
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

from langchain_ollama.chat_models import ChatOllama
from datasets import load_dataset, Dataset

/Users/paul/miniconda3/envs/lc_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
agent_llm = ChatOllama(model="llama3.2",
                      num_predict=1000,
                      top_k=50,
                      top_p=0.95,
                      temperature=0.8,
                      )

In [4]:
class CosState(TypedDict):
    # to track conversation history
    messages: Annotated[list, add_messages]
    # input
    target: str
    statement: str
    # output
    step_one_analysis: str
    step_two_analysis: str
    step_three_analysis: str
    step_four_analysis: str
    step_five_analysis: str
    step_six_analysis: str


class CosInput(TypedDict):
    target: str
    statement: str


class CosOutput(TypedDict):
    step_one_analysis: str
    step_two_analysis: str
    step_three_analysis: str
    step_four_analysis: str
    step_five_analysis: str
    step_six_analysis: str

In [5]:
step_one_template = '''Given text "{statement}", understand the contextual information of the text, which includes the topic of the text, the identity of the author, the target audience, and the relevant socio-cultural background.'''

step_two_template = '''What are the core viewpoints and main intentions to be expressed in the text?'''

step_three_template = '''Analyze the language expression and emotional inclination of the text. Identify the emotional words and rhetorical devices used within the text, and analyze the tone adopted by the author (e.g., affirmative, negative, interrogative, exclamatory, etc.). Based on this analysis, describe the author's emotional attitude.'''

step_four_template = '''Compare similarities and contrasts between text and various possible stances (favor, against, none), based on the above-mentioned information.'''

step_five_template = '''Conduct logical inference based on the context and other relevant information to confirm the consistency and rationality of the stance.'''

step_six_template = '''Based on the above information, determine the stance polarity towards {target}.'''

In [6]:
step_one_prompt = HumanMessagePromptTemplate.from_template(step_one_template)
step_two_prompt = HumanMessagePromptTemplate.from_template(step_two_template)
step_three_prompt = HumanMessagePromptTemplate.from_template(step_three_template)
step_four_prompt = HumanMessagePromptTemplate.from_template(step_four_template)
step_five_prompt = HumanMessagePromptTemplate.from_template(step_five_template)
step_six_prompt = HumanMessagePromptTemplate.from_template(step_six_template)

In [7]:
def agent_node(state: CosState,
               human_msg_template: HumanMessagePromptTemplate,
               agent_name: str) -> CosState:

    if human_msg_template.input_variables:
        human_message = human_msg_template.format(**{k: state[k] for k in human_msg_template.input_variables})
    else:
        human_message = human_msg_template.format()

    res = agent_llm.invoke([*state['messages'], human_message])

    return {f'{agent_name}_analysis': res.content if isinstance(res, BaseMessage) else res,
            'messages': [human_message, res]}

In [8]:
def first_node(state: CosState) -> CosState:
    return agent_node(state, step_one_prompt, 'step_one')

def second_node(state: CosState) -> CosState:
    return agent_node(state, step_two_prompt, 'step_two')

def third_node(state: CosState) -> CosState:
    return agent_node(state, step_three_prompt, 'step_three')

def fourth_node(state: CosState) -> CosState:
    return agent_node(state, step_four_prompt, 'step_four')

def fifth_node(state: CosState) -> CosState:
    return agent_node(state, step_five_prompt, 'step_five')

def sixth_node(state: CosState) -> CosState:
    return agent_node(state, step_six_prompt, 'step_six')

In [9]:
def initialize_graph():
    builder = StateGraph(CosState, input=CosInput, output=CosOutput)

    # Add nodes
    builder.add_node('Context', first_node)
    builder.add_node('MainIdea', second_node)
    builder.add_node('EmotionalAnalysis', third_node)
    builder.add_node('StanceReinforcement', fourth_node)
    builder.add_node('LogicalInference', fifth_node)
    builder.add_node('StanceDeterminism', sixth_node)

    # Add edges
    builder.add_edge(START, 'Context')
    builder.add_edge('Context', 'MainIdea')
    builder.add_edge('MainIdea', 'EmotionalAnalysis')
    builder.add_edge('EmotionalAnalysis', 'StanceReinforcement')
    builder.add_edge('StanceReinforcement', 'LogicalInference')
    builder.add_edge('LogicalInference', 'StanceDeterminism')
    builder.add_edge('StanceDeterminism', END)

    return builder.compile()

In [10]:
graph = initialize_graph()

In [11]:
ds = load_dataset('FundSciImpact/semeval-2016', split='train') 

keep_cols = {'Tweet', 'Target', 'Stance'}
remove_cols = set(ds.column_names) - keep_cols

ds = ds.remove_columns(remove_cols)

def convert_dataset_to_inputs(dataset):
    statements = dataset['Tweet']
    targets = dataset['Target']
    return [{'statement': statement, 
             'target': target} for statement, target in zip(statements, targets)]

inputs = convert_dataset_to_inputs(ds)

In [12]:
test = {"statement": "I love the smell of Hillary in the morning. It smells like Republican Victory. #TedCruz2016 #MarcoRubio",
       "target": "Hillary Clinton"}

In [13]:
graph_output = graph.invoke(test)

In [14]:
graph_output

{'step_one_analysis': 'Based on the given text, here is the contextual information that can be inferred:\n\nTopic: The topic of the text appears to be politics, specifically the 2016 US presidential election.\n\nAuthor Identity: Unfortunately, there is no explicit information about the author\'s identity in the provided text. However, based on the content and tone, it is possible to make some educated guesses.\n\n* The mention of "Hillary" suggests that the author may be referring to Hillary Clinton, a Democratic candidate who ran for president in 2016.\n* The use of "#TedCruz2016" and "#MarcoRubio" also indicates that the author may be supporting Republican candidates, which would suggest an author identity aligned with the Republican Party.\n\nTarget Audience: Based on the tone and language used in the text, it appears to be targeted towards a audience that is likely familiar with US politics, particularly conservative or right-wing individuals. The use of hashtags such as "#TedCruz2